# H² Algebraic Stability Thresholds and Field Extensions

This notebook verifies the algebraic structure of the stability thresholds for
a regular N-vortex ring on the hyperbolic plane H².

**Key formula:** The Riemannian Havelock identity on H² gives eigenvalues
$$\lambda_m \cdot r_E^2 = C_1(\mathrm{H}^2, \xi) - \frac{m(N-m)}{2},$$
where
$$C_1(\mathrm{H}^2, \xi) = \frac{(N-1)(1+\xi^2)}{(1-\xi)^2}, \qquad \xi = r_E^2/a^2.$$

**Key result:** The 7 $\to$ 8 transition threshold is $\xi^* = 8 - 3\sqrt{7}$,
the inverse of the fundamental unit $\varepsilon = 8 + 3\sqrt{7}$ of the ring $\mathbb{Z}[\sqrt{7}]$.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import math
import numpy as np
from fractions import Fraction

from planetary_polygons.extensions.h2_stability import (
    C1_h2_exact, XI_STAR_78, GAMMA_78, threshold_78_exact,
)
from planetary_polygons.extensions.algebraic_thresholds import (
    _squarefree_part,
    h2_threshold_polynomial,
    h2_stability_threshold,
    h2_threshold_table,
)

print('Imports successful.')

## 1. C₁ formula verification

The curvature coefficient on H² is
$$C_1(\mathrm{H}^2, \xi) = \frac{(N-1)(1+\xi^2)}{(1-\xi)^2}.$$

We verify this against a direct manual computation for several values of $N$ and $\xi$.

In [ ]:
print(f'{"N":>3}  {"xi":>8}  {"C1_h2_exact":>14}  {"manual":>14}  {"match":>6}')
print('-' * 55)

test_cases = [
    (5, 0.0),
    (5, 0.1),
    (7, 0.0),
    (7, 0.3),
    (8, 0.0),
    (8, XI_STAR_78),
    (10, 1/7),
    (12, 0.5),
]

for N, xi in test_cases:
    c1_func = C1_h2_exact(N, xi)
    c1_manual = (N - 1) * (1 + xi**2) / (1 - xi)**2
    match = abs(c1_func - c1_manual) < 1e-12
    print(f'{N:>3}  {xi:>8.5f}  {c1_func:>14.8f}  {c1_manual:>14.8f}  {"OK" if match else "FAIL":>6}')

print('\nFlat-space limit check: C1(N, xi->0) = N-1')
for N in [5, 7, 8, 12]:
    c1_flat = C1_h2_exact(N, 1e-15)
    print(f'  N={N}: C1 = {c1_flat:.12f}, N-1 = {N-1}')
    assert abs(c1_flat - (N - 1)) < 1e-10

## 2. The 7 $\to$ 8 transition

In the flat plane, the N-gon is stable for $N \le 7$ (Havelock 1931).
On H², curvature increases $C_1$ beyond $N-1$, extending stability to larger $N$.

The critical threshold for $N=8$ comes from setting
$$C_1(\mathrm{H}^2, \xi^*) = \frac{m(N-m)}{2}\bigg|_{N=8,\,m=4} = 8,$$
which gives the palindromic quadratic
$$\xi^2 - 16\xi + 1 = 0.$$

The smaller root is $\xi^* = 8 - 3\sqrt{7} \approx 0.0627$.

In [ ]:
xi_star, gamma = threshold_78_exact()
print(f'xi*  = {xi_star:.15f}')
print(f'8 - 3*sqrt(7) = {8 - 3*math.sqrt(7):.15f}')
print(f'Match: {abs(xi_star - (8 - 3*math.sqrt(7))) < 1e-14}')
print()

# Verify from the algebraic_thresholds module
xi_star_alg, D, field = h2_stability_threshold(8)
print(f'h2_stability_threshold(8): xi* = {xi_star_alg:.15f}, D = {D}, field = {field}')
print(f'Agreement with XI_STAR_78: {abs(xi_star_alg - XI_STAR_78) < 1e-12}')

In [ ]:
# The key algebraic identity: xi* is the inverse fundamental unit of Z[sqrt(7)]
eps = 8 + 3 * math.sqrt(7)       # fundamental unit
eps_inv = 8 - 3 * math.sqrt(7)   # its inverse

print('Fundamental unit of Z[sqrt(7)]:')
print(f'  epsilon     = 8 + 3*sqrt(7) = {eps:.10f}')
print(f'  epsilon^-1  = 8 - 3*sqrt(7) = {eps_inv:.10f}')
print()

# The norm in Z[sqrt(7)]: N(a + b*sqrt(7)) = a^2 - 7*b^2
norm = 8**2 - 7 * 3**2
product = eps * eps_inv
print(f'Algebraic norm: 8^2 - 7*3^2 = 64 - 63 = {norm}')
print(f'Numerical product: epsilon * epsilon^-1 = {product:.15f}')
print(f'Product = 1: {abs(product - 1.0) < 1e-13}')
print()
print(f'Therefore xi* = epsilon^-1 is the inverse fundamental unit of Z[sqrt(7)].')

In [ ]:
# Verify the palindromic quadratic
A_int, B_int = h2_threshold_polynomial(8)
print(f'Palindromic polynomial for N=8: {A_int}*xi^2 + {B_int}*xi + {A_int} = 0')
print(f'Monic form: xi^2 - {B_int // abs(A_int)}*xi + 1 = 0')
print()

# Check that xi* satisfies it
residual = A_int * xi_star**2 + B_int * xi_star + A_int
print(f'Polynomial residual at xi*: {residual:.2e}')

## 3. Field extension table, $N = 8 \ldots 16$

The marginal stability condition $C_1(\mathrm{H}^2, \xi^*) = m(N-m)/2$ with $m = \lfloor N/2 \rfloor$
gives a palindromic quadratic $A\xi^2 + 2T\xi + A = 0$, with
$$A = (N-1) - \frac{m(N-m)}{2}, \qquad T = \frac{m(N-m)}{2}.$$

The discriminant determines which number field contains $\xi^*$:

| Parity | Pattern | Field |
|--------|---------|-------|
| Even $N$ | $D = \mathrm{sqfree}(N-1)$ | $\mathbb{Q}(\sqrt{D})$ |
| Odd $N$  | $D = \mathrm{sqfree}(N-3)$ | $\mathbb{Q}(\sqrt{D})$ |

**Special case:** $N=10$, where $N-1=9=3^2$ has $\mathrm{sqfree}(9)=1$, giving $\xi^*(10) = 1/7 \in \mathbb{Q}$.

In [ ]:
rows = h2_threshold_table(16)

print(f'{"N":>3}  {"xi*":>12}  {"D":>4}  {"field":>14}  {"polynomial"}')
print('=' * 70)

for row in rows:
    N = row['N']
    xi = row['xi_star']
    D = row['disc_squarefree']
    field = row['field']
    A = row['poly_A']
    B = row['poly_B']
    poly_str = f'{A}*xi^2 + {B}*xi + {A} = 0'
    marker = '  <-- rational!' if D == 1 and N >= 8 else ''
    print(f'{N:>3}  {xi:>12.8f}  {D:>4}  {field:>14}  {poly_str}{marker}')

In [ ]:
# Verify the even/odd field extension pattern
print('Verifying even N pattern: D = squarefree(N-1)')
print('-' * 50)
for row in rows:
    N = row['N']
    if N < 8 or N % 2 != 0:
        continue
    D_actual = row['disc_squarefree']
    D_predicted = _squarefree_part(N - 1)
    ok = D_actual == D_predicted
    print(f'  N={N:>2}: sqfree({N-1:>2}) = {D_predicted:>2}, D = {D_actual:>2}  {"OK" if ok else "FAIL"}')
    assert ok, f'N={N}: pattern mismatch'

print()
print('Verifying odd N pattern: D = squarefree(N-3)')
print('-' * 50)
for row in rows:
    N = row['N']
    if N < 8 or N % 2 != 1:
        continue
    D_actual = row['disc_squarefree']
    D_predicted = _squarefree_part(N - 3)
    ok = D_actual == D_predicted
    print(f'  N={N:>2}: sqfree({N-3:>2}) = {D_predicted:>2}, D = {D_actual:>2}  {"OK" if ok else "FAIL"}')
    assert ok, f'N={N}: pattern mismatch'

print()
print('All field extension patterns verified.')

In [ ]:
# Highlight N=10: unique rational case
xi10, D10, field10 = h2_stability_threshold(10)
print(f'N=10: xi* = {xi10:.15f}')
print(f'       1/7 = {1/7:.15f}')
print(f'   |xi* - 1/7| = {abs(xi10 - 1/7):.2e}')
print(f'   D = {D10}, field = {field10}')
print()
print('N=10 is the unique rational threshold for 8 <= N <= 16')
print('because N-1 = 9 = 3^2 is a perfect square, so sqfree(9) = 1.')
print()

# Verify 1/7 exactly satisfies the polynomial
A10, B10 = h2_threshold_polynomial(10)
print(f'Polynomial for N=10: {A10}*xi^2 + {B10}*xi + {A10} = 0')
xi_frac = Fraction(1, 7)
res_exact = Fraction(A10) * xi_frac**2 + Fraction(B10) * xi_frac + Fraction(A10)
print(f'Exact substitution of xi=1/7: {res_exact}')
assert res_exact == 0, 'Expected exact zero!'

## 4. Numerical verification: $C_1(\mathrm{H}^2, \xi^*) = m(N-m)/2$

At the threshold $\xi^*$, the critical eigenvalue vanishes:
$$C_1(\mathrm{H}^2, \xi^*) = \frac{m(N-m)}{2}, \qquad m = \lfloor N/2 \rfloor.$$

We verify this to high precision for $N = 8, \ldots, 12$.

In [ ]:
print(f'{"N":>3}  {"m":>3}  {"m(N-m)/2":>10}  {"C1(H2, xi*)":>14}  {"residual":>12}')
print('=' * 55)

for N in range(8, 13):
    xi_star_N, _, _ = h2_stability_threshold(N)
    m = N // 2
    T_half = m * (N - m) / 2.0
    C1_val = C1_h2_exact(N, xi_star_N)
    residual = C1_val - T_half
    print(f'{N:>3}  {m:>3}  {T_half:>10.4f}  {C1_val:>14.10f}  {residual:>12.2e}')
    assert abs(residual) < 1e-8, f'N={N}: residual too large!'

print()
print('All marginal conditions satisfied to high precision.')

In [ ]:
# Extended verification: N = 8..16, all thresholds satisfy the marginal condition
print(f'{"N":>3}  {"xi*":>12}  {"C1":>12}  {"T":>8}  {"C1 - T":>12}  {"field":>14}')
print('=' * 72)

for N in range(8, 17):
    xi_star_N, D, field = h2_stability_threshold(N)
    m = N // 2
    T_half = m * (N - m) / 2.0
    C1_val = C1_h2_exact(N, xi_star_N)
    residual = C1_val - T_half
    print(f'{N:>3}  {xi_star_N:>12.8f}  {C1_val:>12.6f}  {T_half:>8.2f}  {residual:>12.2e}  {field:>14}')
    assert abs(residual) < 1e-8

print()
print('All 9 thresholds (N=8..16) verified.')

## 5. Summary

All algebraic thresholds confirmed:

1. **C$_1$ formula:** $C_1(\mathrm{H}^2, \xi) = (N-1)(1+\xi^2)/(1-\xi)^2$ matches manual computation exactly.

2. **7 $\to$ 8 transition:** $\xi^* = 8 - 3\sqrt{7} \approx 0.0627$ is the inverse fundamental unit of $\mathbb{Z}[\sqrt{7}]$, with $(8 - 3\sqrt{7})(8 + 3\sqrt{7}) = 64 - 63 = 1$.

3. **Field extension pattern:** For $N \ge 8$, the threshold $\xi^*(N)$ lies in:
   - Even $N$: $\mathbb{Q}(\sqrt{\mathrm{sqfree}(N-1)})$
   - Odd $N$: $\mathbb{Q}(\sqrt{\mathrm{sqfree}(N-3)})$

4. **Unique rational case:** $N=10$ gives $\xi^*(10) = 1/7 \in \mathbb{Q}$ because $\mathrm{sqfree}(9) = 1$.

5. **Marginal condition:** $C_1(\mathrm{H}^2, \xi^*) = m(N-m)/2$ verified to $< 10^{-8}$ for all $N = 8, \ldots, 16$.